### 1. Imports & Environment Setup
Importing the necessary libraries for data handling (`pandas`, `numpy`), evaluation (`sklearn`), and deep learning (`tensorflow`). We also suppress TensorFlow's background warnings to keep our notebook output clean.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Suppress TensorFlow logging warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout

### 2. Load Data and Train-Test Split
Loading the cleaned dataset and handling any empty text strings. We then split the data into 80% training and 20% testing sets. Using `stratify=df['label']` ensures our train and test sets have a perfectly balanced ratio of fake and real news.

In [2]:
# ==========================================
# 1. Load Data and Train-Test Split
# ==========================================
print("Loading sentiment_dataset.csv...")
df = pd.read_csv('data/cleaned/sentiment_dataset.csv')
df['cleaned_text'] = df['cleaned_text'].fillna('')

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

Loading sentiment_dataset.csv...


### 3. Tokenization and Padding
Neural networks cannot read raw text; they need numbers. We use a `Tokenizer` to convert the top 20,000 most common words into integer IDs. Because neural networks require fixed-size inputs, we use `pad_sequences` to ensure every article is exactly 300 words long (chopping off extra words or adding zeros to short ones).

In [3]:
# ==========================================
# 2. Tokenization and Padding
# ==========================================
print("Tokenizing and padding sequences...")
MAX_VOCAB_SIZE = 20000  # Number of unique words to learn

# --- THE FIX: Data-Driven Sequence Length ---
# Calculate the exact word count of every article in the dataset
article_lengths = X_train_text.apply(lambda x: len(x.split()))

# Find the 95th percentile (This ensures 95% of articles are NOT chopped at all)
optimal_length = int(np.percentile(article_lengths, 95))
MAX_SEQUENCE_LENGTH = optimal_length

print(f"Average article length: {int(article_lengths.mean())} words")
print(f"Maximum article length: {article_lengths.max()} words")
print(f"Optimal sequence length (95th percentile) set to: {MAX_SEQUENCE_LENGTH} words")
# --------------------------------------------

# Tokenizer turns words into integer IDs
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
tokenizer.fit_on_texts(X_train_text)

# Convert text into lists of integers
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post' , truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post' , truncating='post')

Tokenizing and padding sequences...
Average article length: 237 words
Maximum article length: 4854 words
Optimal sequence length (95th percentile) set to: 531 words


### 4. Load Pre-trained GloVe Embeddings
Instead of the model learning word meanings from scratch, we load Stanford's pre-trained GloVe dictionary. We extract the 100-dimensional geometric vectors for our vocabulary and build a custom `embedding_matrix` to feed into our neural network.

In [4]:
# ==========================================
# 3. Load GloVe Embeddings
# ==========================================
print("Loading GloVe word embeddings...")
EMBEDDING_DIM = 100
embeddings_index = {}

# Make sure glove.6B.100d.txt is in your data directory!
with open('data/glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

# Create an embedding matrix for the words actually in our dataset
word_index = tokenizer.word_index
embedding_matrix = np.zeros((MAX_VOCAB_SIZE, EMBEDDING_DIM))

for word, i in word_index.items():
    if i < MAX_VOCAB_SIZE:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            # Words not found in GloVe will remain all zeros
            embedding_matrix[i] = embedding_vector

Loading GloVe word embeddings...
Found 400000 word vectors.


### 5. Build the Feed-Forward Neural Network (MLP)
Constructing the architecture of the model. 
1. **Embedding Layer:** Translates our tokenized words into GloVe vectors.
2. **Pooling Layer:** Mathematically averages all 300 word vectors into a single "summary" vector for the entire article.
3. **Dense Layers:** Standard decision-making layers with `Dropout` to randomly turn off neurons, preventing the model from memorizing the data (overfitting).

In [5]:
# ==========================================
# 4. Build the Feed-Forward Neural Network (MLP)
# ==========================================
print("Building the MLP Model...")
model = Sequential()

# 1. Embedding Layer: Translates word IDs into GloVe geometric vectors
model.add(Embedding(
    input_dim=MAX_VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    mask_zero=True,
    trainable=False  # Freeze GloVe weights so they aren't destroyed during early training
))

# 2. GlobalAveragePooling1D Layer: Averages all word vectors in the sequence
#    into a single fixed-length vector (replaces the recurrent BiLSTM layer)
model.add(GlobalAveragePooling1D())

# 3. Dense & Dropout Layers: For decision making and preventing overfitting
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))  # Randomly turns off 50% of neurons to prevent memorizing the data

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))  # Randomly turns off 50% of neurons to prevent memorizing the data

# 4. Output Layer: Compresses output to a probability between 0 and 1
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Building the MLP Model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,000,000 (7.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,000,000 (7.63 MB)

### 6. Train and Evaluate
Training the model on our padded sequences in batches. We use a 10% validation split during training to monitor for overfitting. Finally, we make predictions on our unseen test set and output the Accuracy and F1-Score to compare against our classical ML models.

In [6]:
# ==========================================
# 5. Train and Evaluate
# ==========================================
print("Training model... (This will take a few minutes)")

# We use validation_split to monitor for overfitting during training
X_train_final, X_val_pad, y_train_final, y_val = train_test_split(
    X_train_pad, y_train, test_size=0.1, random_state=42, stratify=y_train
)

history = model.fit(X_train_final, y_train_final, epochs=5, batch_size=64, validation_data=(X_val_pad, y_val))



print("\nEvaluating on Test Set...")

# Neural networks output probabilities (e.g., 0.89). We convert > 0.5 to 1 (Real), else 0 (Fake)
y_pred_probs = model.predict(X_test_pad)
y_pred = (y_pred_probs > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\n--- MLP Evaluation ---")
print(f"Accuracy: {acc:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Fake (0)', 'Real (1)']))

Training model... (This will take a few minutes)
Epoch 1/5
505/505 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8644 - loss: 0.3186 - val_accuracy: 0.9243 - val_loss: 0.2014
Epoch 2/5
505/505 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9222 - loss: 0.2055 - val_accuracy: 0.9321 - val_loss: 0.1783
Epoch 3/5
505/505 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9304 - loss: 0.1854 - val_accuracy: 0.9368 - val_loss: 0.1661
Epoch 4/5
505/505 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9357 - loss: 0.1761 - val_accuracy: 0.9407 - val_loss: 0.1606
Epoch 5/5
505/505 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9381 - loss: 0.1682 - val_accuracy: 0.9415 - val_loss: 0.1533

Evaluating on Test Set...
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

--- MLP Evaluation ---
Accuracy: 0.9450
F1-Score: 0.9424

Classification Report:
              precision    recall  f1-score   support

    Fake (0)       0.95      0.95      0.95      4695
    Real (1)       0.94      0.94      0.94      4283

 